In [1]:
!pip install --upgrade google-cloud-storage pandas scikit-learn gcsfs

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from google.cloud import storage

from google.colab import auth
auth.authenticate_user()

In [3]:
PROJECT_ID = "project-2a282f15-18ae-4b89-a5f"
BUCKET_NAME = "conor-weekes-data-20084449"
LOCATION = "EU"

In [4]:
client = storage.Client(project=PROJECT_ID)

print("Files in bucket:")

for blob in client.list_blobs(BUCKET_NAME):
    print(blob.name)

Files in bucket:
processed/
raw/
raw/MutualFunds.csv
titanic.csv


In [5]:
GCS_PATH = f"gs://{BUCKET_NAME}/raw/MutualFunds.csv"

df = pd.read_csv(GCS_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (23783, 298)


,fund_symbol,quote_type,region,fund_short_name,fund_long_name,currency,initial_investment,subsequent_investment,fund_category,fund_family,...,peer_environment_avg,peer_environment_max,social_score,peer_social_min,peer_social_avg,peer_social_max,governance_score,peer_governance_min,peer_governance_avg,peer_governance_max
0,AAAAX,MutualFund,US,DWS RREEF Real Assets Fund - Cl,DWS RREEF Real Assets Fund - Class A,USD,1000.0,50.0,World Allocation,DWS,...,5.05,10.58,7.43,5.98,9.07,11.30,5.43,4.26,7.14,8.11
1,AAAEX,MutualFund,US,AllianzGI Health Sciences Fund,Virtus AllianzGI Health Sciences Fund Class P,USD,1000000.0,NaN,Health,Virtus,...,1.43,3.27,12.96,9.52,12.87,15.08,8.40,4.96,7.68,10.30
2,AAAFX,MutualFund,US,NaN,American Century One Choice Blend+ 2015 Portfo...,USD,2500.0,50.0,Target-Date 2015,American Century Investments,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAAGX,MutualFund,US,Thrivent Large Cap Growth Fund,Thrivent Large Cap Growth Fund Class A,USD,2000.0,50.0,Large Growth,Thrivent Funds,...,2.70,5.81,10.13,7.25,10.14,11.97,8.03,5.30,7.54,8.90
4,AAAHX,MutualFund,US,NaN,American Century One Choice Blend+ 2015 Portfo...,USD,5000000.0,NaN,Target-Date 2015,American Century Investments,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The following 4 checks were completed to access if this dataset could be used for the project but I thought it would be useful to leave these lines in to see how I came to my conclusion even if they are not necessary to reproduce the processed data set.


In [6]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
for col in df.columns:
    print(col)

Number of rows: 23783
Number of columns: 298

Column names:
fund_symbol
quote_type
region
fund_short_name
fund_long_name
currency
initial_investment
subsequent_investment
fund_category
fund_family
exchange_code
exchange_name
exchange_timezone
management_name
management_bio
management_start_date
total_net_assets
year_to_date_return
day50_moving_average
day200_moving_average
week52_high_low_change
week52_high_low_change_perc
week52_high
week52_high_change
week52_high_change_perc
week52_low
week52_low_change
week52_low_change_perc
investment_strategy
fund_yield
morningstar_overall_rating
morningstar_risk_rating
inception_date
last_dividend
last_cap_gain
annual_holdings_turnover
investment_type
size_type
fund_annual_report_net_expense_ratio
category_annual_report_net_expense_ratio
fund_prospectus_net_expense_ratio
fund_prospectus_gross_expense_ratio
fund_max_12b1_fee
fund_max_front_end_sales_load
category_max_front_end_sales_load
fund_max_deferred_sales_load
category_max_deferred_sales_loa

In [7]:
print("Duplicate rows:", df.duplicated().sum())

missing = df.isnull().mean().sort_values(ascending=False)

print("\nColumns with the most missing data:")
display(missing.head(20))

Duplicate rows: 0

Columns with the most missing data:


last_cap_gain                    0.999706
fund_return_2021_q3              0.997393
fund_max_deferred_sales_load     0.873733
fund_return_2021_q2              0.847917
fund_max_front_end_sales_load    0.844721
fund_return_2000_q4              0.833663
fund_return_2000                 0.833663
fund_return_2000_q3              0.829752
fund_return_2000_q2              0.824076
fund_return_2000_q1              0.818316
fund_return_2001                 0.810579
fund_return_2001_q4              0.810579
fund_return_2001_q3              0.801034
fund_return_2001_q2              0.795778
fund_return_2001_q1              0.790354
fund_return_2002_q4              0.785267
fund_return_2002                 0.785141
fund_return_2002_q3              0.778245
fund_return_2002_q2              0.770382
fund_return_2002_q1              0.763529
dtype: float64

In [8]:
print("Fund category values:")
print(df["fund_category"].value_counts().head(20))

print("\nNumber of different fund categories:")
print(df["fund_category"].nunique())

print("\nMissing fund categories:")
print(df["fund_category"].isnull().sum())

Fund category values:
fund_category
Large Blend                      1235
Large Growth                     1172
Large Value                      1080
Diversified Emerging Mkts         741
Foreign Large Blend               680
Allocation--50% to 70% Equity     658
High Yield Bond                   629
Small Blend                       579
Small Growth                      578
Short-Term Bond                   570
Intermediate Core-Plus Bond       566
Mid-Cap Growth                    562
Allocation--30% to 50% Equity     491
World Allocation                  448
Foreign Large Growth              431
Intermediate Core Bond            413
Small Value                       409
Mid-Cap Value                     368
Multisector Bond                  351
Nontraditional Bond               342
Name: count, dtype: int64

Number of different fund categories:
119

Missing fund categories:
663


In [9]:
category_counts = df["fund_category"].value_counts()

print("Smallest categories:")
display(category_counts.tail(20))

print("Categories with fewer than 10 funds:")
print((category_counts < 10).sum())

Smallest categories:


fund_category
Long Government                 23
Relative Value Arbitrage        20
Japan Stock                     19
Consumer Cyclical               19
Industrials                     17
Latin America Stock             17
Trading--Inverse Debt           14
Consumer Defensive              12
India Equity                    11
Trading--Miscellaneous           9
Muni Target Maturity             9
Trading--Leveraged Debt          8
World Large Stock                5
Diversified Pacific/Asia         5
Miscellaneous Sector             4
Multialternative                 4
Trading--Inverse Commodities     4
Market Neutral                   1
Long-Short Credit                1
World Large-Stock Growth         1
Name: count, dtype: int64

Categories with fewer than 10 funds:
11


In [ ]:
# Remove rows where fund_category is missing
df_clean = df.dropna(subset=["fund_category"]).copy()

# Count each category
category_counts = df_clean["fund_category"].value_counts()

# Keep categories with at least 10 funds
valid_categories = category_counts[category_counts >= 10].index

df_clean = df_clean[
    df_clean["fund_category"].isin(valid_categories)
].copy()

print("Rows remaining:", len(df_clean))
print("Categories remaining:", df_clean["fund_category"].nunique())

Rows remaining: 23069
Categories remaining: 108


In [ ]:
# Sources Used:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html
# https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isin.html



In [11]:
train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.20,
    random_state=42,
    stratify=df_clean["fund_category"]
)

dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["fund_category"]
)

print("Train:", len(train_df))
print("Dev:", len(dev_df))
print("Test:", len(test_df))

Train: 18455
Dev: 2307
Test: 2307


#Source Used:
#https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

In [12]:
distribution = pd.DataFrame({
    "train": train_df["fund_category"].value_counts(normalize=True),
    "dev": dev_df["fund_category"].value_counts(normalize=True),
    "test": test_df["fund_category"].value_counts(normalize=True)
})

display(distribution.head(20))

,train,dev,test
fund_category,,,
Allocation--15% to 30% Equity,0.007911,0.007802,0.007802
Allocation--30% to 50% Equity,0.021295,0.021240,0.021240
Allocation--50% to 70% Equity,0.028502,0.028609,0.028609
Allocation--70% to 85% Equity,0.012246,0.012137,0.012137
Allocation--85%+ Equity,0.006773,0.006935,0.006502
Bank Loan,0.009862,0.009970,0.009970
China Region,0.003576,0.003468,0.003468
Commodities Broad Basket,0.003847,0.003901,0.003901
Communications,0.001355,0.001300,0.001300


#Source Used:
#https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

skf = StratifiedKFold(
    n_splits=8,
    shuffle=True,
    random_state=42
)

cv_folds = {}

for fold_idx, (train_idx, val_idx) in enumerate(
    skf.split(train_df, train_df["fund_category"])
):
    cv_folds[fold_idx] = (
        train_df.iloc[train_idx].copy(),
        train_df.iloc[val_idx].copy()
    )

print("Number of CV folds:", len(cv_folds))

Number of CV folds: 8


#https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html

In [14]:
import os

os.makedirs("shards/cv_folds", exist_ok=True)

# Save train, dev and test
train_df.to_csv("shards/train.csv", index=False)
dev_df.to_csv("shards/dev.csv", index=False)
test_df.to_csv("shards/test.csv", index=False)

# Save the 8 cross-validation folds
for fold_idx, (f_train, f_val) in cv_folds.items():
    f_train.to_csv(f"shards/cv_folds/fold_{fold_idx}_train.csv", index=False)
    f_val.to_csv(f"shards/cv_folds/fold_{fold_idx}_val.csv", index=False)

print("All shard files saved locally.")

All shard files saved locally.


In [ ]:
def upload_folder_to_gcs(local_folder, bucket_name, gcs_prefix):
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    for root, _, files in os.walk(local_folder):
        for file in files:
            local_path = os.path.join(root, file)

            relative_path = os.path.relpath(
                local_path,
                local_folder
            )

            gcs_path = os.path.join(
                gcs_prefix,
                relative_path
            ).replace("\\", "/")

            blob = bucket.blob(gcs_path)
            blob.upload_from_filename(local_path)

            print("Uploaded:", gcs_path)


upload_folder_to_gcs(
    "shards",
    BUCKET_NAME,
    "sharded"
)

Uploaded: sharded/dev.csv
Uploaded: sharded/test.csv
Uploaded: sharded/train.csv
Uploaded: sharded/cv_folds/fold_5_val.csv
Uploaded: sharded/cv_folds/fold_7_train.csv
Uploaded: sharded/cv_folds/fold_4_val.csv
Uploaded: sharded/cv_folds/fold_0_train.csv
Uploaded: sharded/cv_folds/fold_3_val.csv
Uploaded: sharded/cv_folds/fold_3_train.csv
Uploaded: sharded/cv_folds/fold_6_train.csv
Uploaded: sharded/cv_folds/fold_0_val.csv
Uploaded: sharded/cv_folds/fold_4_train.csv
Uploaded: sharded/cv_folds/fold_2_train.csv
Uploaded: sharded/cv_folds/fold_1_train.csv
Uploaded: sharded/cv_folds/fold_5_train.csv
Uploaded: sharded/cv_folds/fold_2_val.csv
Uploaded: sharded/cv_folds/fold_6_val.csv
Uploaded: sharded/cv_folds/fold_7_val.csv
Uploaded: sharded/cv_folds/fold_1_val.csv


#Source Used:
#https://docs.python.org/3/library/os.html
